# **KKBOX Churn Prediction And Retention Intelligence System - Data Preparation and Feature Ingineering**

## **1. Objectives**

## **2. Load Data & Basic Setup**

In [24]:

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


current_dir = os.getcwd()


while not os.path.exists(os.path.join(current_dir, "data", "raw")):
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        raise FileNotFoundError("Project root with data/raw not found")
    current_dir = parent


os.chdir(current_dir)

print("Project root:", os.getcwd())
print("data/raw exists:", os.path.exists("data/raw"))


pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

target = "is_churn"

Project root: c:\Users\pauli\OneDrive\Documentos\GitHub\customer-churn-intelligence-system
data/raw exists: True


## **3. Light Cleaning**

In [25]:
# 3.1. Train dataset

train = pd.read_csv("data/raw/train_v2.csv")
train = train.drop_duplicates(subset="msno")

train["is_churn"] = train["is_churn"].astype(int)

In [26]:
# 3.2. Members dataset
members = pd.read_csv("data/raw/members_v3.csv")

members = members.drop_duplicates(subset="msno")

members.loc[(members["bd"] < 0) | (members["bd"] > 100), "bd"] = np.nan

members["gender"] = members["gender"].astype("category")

members["registration_init_time"] = pd.to_datetime(
    members["registration_init_time"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

members["city"] = pd.to_numeric(members["city"], errors="coerce")
members["registered_via"] = pd.to_numeric(members["registered_via"], errors="coerce")


In [27]:
# 3.3. Transactions dataset
transactions = pd.read_csv("data/raw/transactions_v2.csv")

transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

transactions["membership_expire_date"] = pd.to_datetime(
    transactions["membership_expire_date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

transactions = transactions[
    transactions["transaction_date"] <= transactions["membership_expire_date"]
]

num_cols = [
    "payment_method_id", "payment_plan_days", "plan_list_price",
    "actual_amount_paid", "is_auto_renew", "is_cancel"
]

transactions[num_cols] = transactions[num_cols].apply(pd.to_numeric, errors="coerce")

transactions.loc[transactions["actual_amount_paid"] < 0, "actual_amount_paid"] = np.nan
transactions.loc[transactions["plan_list_price"] < 0, "plan_list_price"] = np.nan
transactions.loc[transactions["payment_plan_days"] <= 0, "payment_plan_days"] = np.nan


In [28]:
# 3.4. User Logs dataset
user_logs = pd.read_csv("data/raw/user_logs_v2.csv")

log_cols = [
    "num_25", "num_50", "num_75", "num_985",
    "num_100", "num_unq", "total_secs"
]

user_logs[log_cols] = user_logs[log_cols].apply(pd.to_numeric, errors="coerce")

user_logs.loc[user_logs["total_secs"] < 0, "total_secs"] = np.nan
user_logs.loc[user_logs["num_unq"] < 0, "num_unq"] = np.nan

user_logs["date"] = pd.to_datetime(
    user_logs["date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

user_logs = user_logs[
    (user_logs["date"] >= pd.Timestamp("2010-01-01")) &
    (user_logs["date"] <= pd.Timestamp("2020-01-01"))
]


In [ ]:
# 3.5. Quick Sanity Checks

print("Train:", train.shape)
print("Members:", members.shape)
print("Transactions:", transactions.shape)
print("User logs:", user_logs.shape)

print("Missing values in train:\n", train.isnull().mean().sort_values(ascending=False).head(10))
print("Missing values in members:\n", members.isnull().mean().sort_values(ascending=False).head(10))
print("Missing values in transactions:\n", transactions.isnull().mean().sort_values(ascending=False).head(10))
print("Missing values in user_logs:\n", user_logs.isnull().mean().sort_values(ascending=False).head(10))


print("Duplicate users in train:", train["msno"].duplicated().sum())
print("Duplicate users in members:", members["msno"].duplicated().sum())
print("Exact duplicate rows in transactions:", transactions.duplicated().sum())
print("Exact duplicate rows in user_logs:", user_logs.duplicated().sum())

train = train.reset_index(drop=True)
members = members.reset_index(drop=True)
transactions = transactions.reset_index(drop=True)
user_logs = user_logs.reset_index(drop=True)

Train: (970960, 2)
Members: (6769473, 6)
Transactions: (1425903, 9)
User logs: (18396362, 9)
Missing values in train:
 msno        0.0
is_churn    0.0
dtype: float64
Missing values in members:
 gender                    0.654335
bd                        0.000835
city                      0.000000
msno                      0.000000
registered_via            0.000000
registration_init_time    0.000000
dtype: float64
Missing values in transactions:
 payment_plan_days         0.001556
msno                      0.000000
payment_method_id         0.000000
plan_list_price           0.000000
actual_amount_paid        0.000000
is_auto_renew             0.000000
transaction_date          0.000000
membership_expire_date    0.000000
is_cancel                 0.000000
dtype: float64
Missing values in user_logs:
 msno          0.0
date          0.0
num_25        0.0
num_50        0.0
num_75        0.0
num_985       0.0
num_100       0.0
num_unq       0.0
total_secs    0.0
dtype: float64
Duplicate u

## **4. Aggregate Transactions & Usaer Logs**

In [29]:
# 4.1.  Aggregate Transactions (from transaction-level to user-level)

# Aggregate transactions to customer level
transactions_agg = transactions.groupby("msno").agg({
    "actual_amount_paid": ["sum", "mean"],
    "payment_plan_days": ["sum", "mean"],
    "plan_list_price": "mean",
    "is_auto_renew": ["mean", "max"],
    "is_cancel": ["mean", "max"],
    "transaction_date": ["count", "min", "max"],
    "membership_expire_date": "max"
}).reset_index()

# Flatten multi-index column names
transactions_agg.columns = [
    "msno",
    "total_amount_paid",
    "avg_amount_paid",
    "total_payment_plan_days",
    "avg_payment_plan_days",
    "avg_plan_list_price",
    "auto_renew_rate",
    "has_auto_renew",
    "cancel_rate",
    "has_cancelled",
    "transaction_count",
    "first_transaction_date",
    "last_transaction_date",
    "membership_expire_date"
]

print("Transactions aggregated:", transactions_agg.shape)
transactions_agg.head()

Transactions aggregated: (1194686, 14)


,msno,total_amount_paid,avg_amount_paid,total_payment_plan_days,avg_payment_plan_days,avg_plan_list_price,auto_renew_rate,has_auto_renew,cancel_rate,has_cancelled,transaction_count,first_transaction_date,last_transaction_date,membership_expire_date
0,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,1599.0,1599.0,395.0,395.0,1599.0,0.0,0,0.0,0,1,2016-10-23,2016-10-23,2018-02-06
1,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,99.0,99.0,30.0,30.0,99.0,1.0,1,0.0,0,1,2017-03-15,2017-03-15,2017-04-15
2,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,298.0,149.0,60.0,30.0,149.0,1.0,1,0.0,0,2,2017-02-28,2017-03-31,2017-05-19
3,+++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc=,149.0,149.0,30.0,30.0,149.0,1.0,1,0.0,0,1,2017-03-26,2017-03-26,2017-04-26
4,++/9R3sX37CjxbY/AaGvbwr3QkwElKBCtSvVzhCBDOk=,149.0,149.0,30.0,30.0,149.0,1.0,1,0.0,0,1,2017-03-15,2017-03-15,2017-04-15


In [30]:
# 4.2. Aggregate User Logs (from activity-level to user-level)

user_logs_agg = user_logs.groupby("msno").agg({
    "total_secs": "sum",
    "num_unq": "sum"
}).reset_index()

print("User logs aggregated:", user_logs_agg.shape)
user_logs_agg.head()


User logs aggregated: (1103894, 3)


,msno,total_secs,num_unq
0,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,117907.425,530.0
1,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,192527.892,885.0
2,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,115411.260,468.0
3,+++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc=,149896.558,828.0
4,++/9R3sX37CjxbY/AaGvbwr3QkwElKBCtSvVzhCBDOk=,116433.247,230.0


## **5. Merge**

In [31]:
# 5.1. Merge all datasets into a single modeling table

df = (
    train
    .merge(members, on="msno", how="left")
    .merge(transactions_agg, on="msno", how="left")
    .merge(user_logs_agg, on="msno", how="left")
)

print("Final dataset shape:", df.shape)
df.head()

Final dataset shape: (970960, 22)


,msno,is_churn,city,bd,gender,registered_via,registration_init_time,total_amount_paid,avg_amount_paid,total_payment_plan_days,avg_payment_plan_days,avg_plan_list_price,auto_renew_rate,has_auto_renew,cancel_rate,has_cancelled,transaction_count,first_transaction_date,last_transaction_date,membership_expire_date,total_secs,num_unq
0,ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=,1,5.0,28.0,male,3.0,2013-12-23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaT,80598.557,348.0
1,f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=,1,13.0,20.0,male,3.0,2013-12-23,180.0,180.0,30.0,30.0,180.0,0.0,0.0,0.000,0.0,1.0,2017-03-11,2017-03-11,2017-04-11,6986.509,30.0
2,zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=,1,13.0,18.0,male,3.0,2013-12-27,300.0,150.0,150.0,75.0,150.0,0.0,0.0,0.000,0.0,2.0,2017-03-11,2017-03-14,2017-06-15,67810.467,432.0
3,8iF/+8HY8lJKFrTc7iR9ZYGCG2Ecrogbc2Vy5YhsfhQ=,1,1.0,0.0,NaN,7.0,2014-01-09,1490.0,149.0,300.0,30.0,149.0,1.0,1.0,0.000,0.0,10.0,2015-08-08,2015-12-08,2018-01-08,NaN,NaN
4,K6fja4+jmoZ5xG6BypqX80Uw/XKpMgrEMdG2edFOxnA=,1,13.0,35.0,female,7.0,2014-01-25,792.0,99.0,240.0,30.0,99.0,1.0,1.0,0.125,1.0,8.0,2016-10-01,2017-03-16,2017-09-18,239882.241,548.0


In [33]:
# 5.2. Quick Sanity Check after merge

print("Missing values after merge:")
print(df.isnull().mean().sort_values(ascending=False).head(20))

print("Duplicate users in final df:", df["msno"].duplicated().sum())



Missing values after merge:
gender                     0.599463
num_unq                    0.222881
total_secs                 0.222881
bd                         0.113821
city                       0.113283
registered_via             0.113283
registration_init_time     0.113283
avg_payment_plan_days      0.040735
has_auto_renew             0.040734
auto_renew_rate            0.040734
total_payment_plan_days    0.040734
avg_plan_list_price        0.040734
avg_amount_paid            0.040734
total_amount_paid          0.040734
first_transaction_date     0.040734
transaction_count          0.040734
cancel_rate                0.040734
has_cancelled              0.040734
membership_expire_date     0.040734
last_transaction_date      0.040734
dtype: float64
Duplicate users in final df: 0
